## Downsampling Algorithm for EIS data

There are a variety of statistical methods for downsampling timeseries data of various shapes, and yet the peculiar nature of electrochemical impedance spectroscopy data presented an opportunity for further developmment.

Window functions like simple moving averages or largest triangle three buckets are great for densely populated polynomials or otherwise uniformly populated datasets, but fail when considering large enough datasets with extreme minutia, like sparse asymptotes.

### Background and Descriptions

This algorithm targets galvanostatic test data for electrolysis stacks. The galvanostatic test reads data at a low interval, typically just a few seconds, for thousands of hours. The measurements taken are voltage, and current, among other values associated with EIS data.

#### Voltage data
Voltage data is characterized by several segments of data with a slope approximating infinity as a voltage is applied across the electrolysis stack. The performance of the stack degrades over time, showing a slightly negative trend, with many samples also taken at higher or lower voltages.

#### Current data
Current data is characterized by an initial ramp up, and many millions of readings at a constant current. The current changes based on the state of the system, for example whether the stack is heating up, or on load.

### Downsampling

The literature describe a few algoritms designed to downsample timeseries data while preserving their shape. First, the Ramer–Douglas–Peucker algorithm simplifies curves by decimating curves into representative line segments that result in a curve with fewer points. This algorithm however failed to account for the distinct asymptotes in EIS data.

Recent advances in downsampling algorithms for timeseries data from industrial processes have similarly developed a methodology for intelligent downsampling. The multivariate adaptive downsampling algorithm was replicated in Python and applied to sample EIS data, but because EIS data is so densely populated, it fails to interpret any distinct time domains where variance in the data exist. 

## Results
This algorithm uses the `np.diff()` function to calculate relative outliers in the data, neatly identifying variance like asymptotes in EIS data. The algorithm ranks the outliers and takes a fraction of them, before iterating through the remainder of the dataset and taking samples.

The histograms show that the original and downsampled dataset are statistically similar, and a visual analysis is made possible by plotting the original and downsampled timeseries data together.

In [ ]:
##############################
# DOWNSAMPLING
##############################

import json
import os
import numpy as np
import time as timer

# Target directory for downsampling
fixture_dir = "./FCE/stack1"

original = []

time = []
voltage = []
current = []

downsampled_time = np.array([])
downsampled_current = np.array([])
downsampled_voltage = np.array([])

# Downsample to 10,000 points
num_files = len(os.listdir(fixture_dir))
n = 10000

for filename in os.listdir(fixture_dir):

    start_time = timer.time()
    
    if filename.endswith(".json") and filename != "downsample.json":
        with open(os.path.join(fixture_dir, filename), 'r') as f:
            fixtures = json.load(f)

            for obj in fixtures:
                original.append(obj)
                fields = obj['fields']
                data = fields['data']
                time.append(data['time'])
                voltage.append(data['anonymized_voltage'])
                current.append(data['anonymized_current'])
                
                # Delete private fields
                del data['current']
                del data['voltage']

        end_time = timer.time()
        print(f"Processed {filename} in {end_time - start_time} seconds")

In [350]:
n = 10000

anonymized_time = np.array(time)
anonymized_voltage = np.array(voltage)
anonymized_current = np.array(current)

# Detect changes in current and voltage data
voltage_diff = np.abs(np.diff(anonymized_voltage))
current_diff = np.abs(np.diff(anonymized_current))

# Find outliers by taking the top percentile changes
voltage_outliers = np.where(voltage_diff > np.percentile(voltage_diff, 90))[0] + 1
current_outliers = np.where(current_diff > np.percentile(current_diff, 99))[0] + 1
outliers = np.unique(np.concatenate([voltage_outliers, current_outliers]))

# Select the n // 5 most significant outliers
if len(outliers) > 0:
    significance = np.amax([np.abs(np.diff(anonymized_voltage[outliers])), np.abs(np.diff(anonymized_current[outliers]))], axis=0)
    outliers = outliers[np.argsort(significance)][-(n//5):]

# Evenly sample m records to achieve a total of n
m = n - len(outliers)

if len(outliers) < n:
    m = n - len(outliers)

    # Mask over indices that are already outliers
    mask = np.ones(len(time), dtype=bool)
    mask[outliers] = False
    indices = np.where(mask)[0]

    # Interpolate over the remaining set of data
    if len(indices) > 1:
        interp = np.linspace(0, len(indices) - 1, m, dtype=int)
        interp = indices[interp]
    else:
        interp = np.array([], dtype=int)
    
    indices = np.concatenate([outliers, interp])


# Extract downsampled time
downsampled_time = anonymized_time[indices]
downsampled_voltage = anonymized_voltage[indices]
downsampled_current = anonymized_current[indices]

## Histograms

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot the histogram of anonymized_voltage in the first subplot
n, bins, patches = axs[0].hist(anonymized_voltage, bins=30, density=True, alpha=0.6, color='olive', label='Distribution')
axs[0].set_title('Voltage')
axs[0].set_xlabel('Value')
axs[0].set_ylabel('Probability Density')
axs[0].legend()

# Plot the histogram of downsampled_voltage in the second subplot
n, bins, patches = axs[1].hist(downsampled_voltage, bins=30, density=True, alpha=0.6, color='black', label='Distribution')
axs[1].set_title('Downsampled Voltage')
axs[1].set_xlabel('Value')
axs[1].set_ylabel('Probability Density')
axs[1].legend()

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot the histogram of anonymized_voltage in the first subplot
n, bins, patches = axs[0].hist(anonymized_current, bins=30, density=True, alpha=0.6, color='olive', label='Distribution')
axs[0].set_title('Current')
axs[0].set_xlabel('Value')
axs[0].set_ylabel('Probability Density')
axs[0].legend()

# Plot the histogram of downsampled_voltage in the second subplot
n, bins, patches = axs[1].hist(downsampled_current, bins=30, density=True, alpha=0.6, color='black', label='Distribution')
axs[1].set_title('Downsampled Current')
axs[1].set_xlabel('Value')
axs[1].set_ylabel('Probability Density')
axs[1].legend()

## Visualization

In [ ]:
# Voltage Comparison
plt.figure(figsize=(12, 6))
plt.scatter(anonymized_time, anonymized_voltage, label='Voltage', color='olive', alpha=1, s=10)
plt.scatter(downsampled_time, downsampled_voltage, label="Downsampled Voltage", color='black', alpha=1, s=10)
plt.legend()
plt.show()

In [ ]:
# Current Comparison
plt.figure(figsize=(12, 6))
plt.scatter(anonymized_time, anonymized_current, label='Current', color='olive', alpha=0.5, s=10)
plt.scatter(downsampled_time, downsampled_current, label="Downsampled Current", color='black', alpha=1, s=10)
plt.legend()
plt.show()

In [356]:
original = np.array(original)
keep = original[indices]

with open(fixture_dir + "/" + "downsample.json", "w") as file:
    json.dump(list(keep), file, indent=4)